##Load all these

In [ ]:
import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Mount Drive (uncomment if needed)
# from google.colab import drive
# drive.mount('/content/drive')

# Load Keras models
model_rnn = load_model('/content/drive/My Drive/model_rnn.keras')
model_lstm = load_model('/content/drive/My Drive/model_lstm.keras')
model_gru = load_model('/content/drive/My Drive/model_gru.keras')
model_bilstm = load_model('/content/drive/My Drive/model_bilstm.keras')

# Load histories
with open('/content/drive/My Drive/histories.pkl', 'rb') as f:
    histories = pickle.load(f)

class History:
    def __init__(self, hist):
        self.history = hist

history_rnn = History(histories['rnn'])
history_lstm = History(histories['lstm'])
history_gru = History(histories['gru'])
history_bilstm = History(histories['bilstm'])

# Load variables
with open('/content/drive/My Drive/variables.pkl', 'rb') as f:
    vars_dict = pickle.load(f)

vocab_size = vars_dict['vocab_size']
max_len = vars_dict['max_len']
rnn_time = vars_dict['rnn_time']
lstm_time = vars_dict['lstm_time']
gru_time = vars_dict['gru_time']
bilstm_time = vars_dict['bilstm_time']
bert_time = vars_dict['bert_time']

# Load results
results_df = pd.read_csv('/content/drive/My Drive/final_results.csv')

# Load predictions
with open('/content/drive/My Drive/predictions.pkl', 'rb') as f:
    preds = pickle.load(f)

# Load data (if needed for re-evaluation)
# df_balanced = pd.read_csv('/content/drive/My Drive/df_balanced_v2.csv')
# with open('/content/drive/My Drive/X_train_pad_v2.pkl', 'rb') as f:
#     X_train_pad = pickle.load(f)
# with open('/content/drive/My Drive/X_test_pad_v2.pkl', 'rb') as f:
#     X_test_pad = pickle.load(f)
# with open('/content/drive/My Drive/y_train_v2.pkl', 'rb') as f:
#     y_train = pickle.load(f)
# with open('/content/drive/My Drive/y_test_v2.pkl', 'rb') as f:
#     y_test = pickle.load(f)

print("All models and data loaded successfully!")
print(f"\nResults Summary:")
print(results_df.to_string(index=False))

# Sentiment Analysis of Hinglish YouTube Comments Using Deep Learning

## Abstract

Sentiment analysis is a core problem in natural language processing with applications in recommendation systems, market analysis, and social media monitoring. This research develops and evaluates four deep learning architectures — RNN, LSTM, GRU, and BiLSTM — for sentiment classification of code-mixed Hinglish YouTube comments. A transformer baseline (DistilBERT) is included for comparative context. Performance is assessed across accuracy, precision, recall, F1-score, training time, and memory usage. The study provides a reproducible, multi-dimensional assessment of sequence models for practical deployment.

**Keywords:** Sentiment Analysis, Hinglish, RNN, LSTM, GRU, BiLSTM, DistilBERT, YouTube Comments

In [ ]:
!pip install youtube-comment-downloader

## 1. Data Collection

### 1.1 Source Selection

YouTube comments were chosen over standard datasets (IMDB, Twitter) for three reasons:

- **Code-mixed language:** Hinglish (Hindi + English) is understudied in NLP literature
- **Real-world noise:** Comments contain slang, typos, emojis, and informal grammar
- **Domain relevance:** Video comments reflect spontaneous, emotional reactions

### 1.2 Video Selection

Twelve videos were selected across entertainment, music, and vlog categories to ensure diverse sentiment distribution. The `youtube-comment-downloader` library was used with `SORT_BY_POPULAR` to capture the most engaging (and thus sentiment-rich) comments.

**Collection parameters:**
- 2,000 comments per video
- 24,000 total comments
- Popular sort order for representativeness

In [ ]:
from youtube_comment_downloader import YoutubeCommentDownloader, SORT_BY_POPULAR
import pandas as pd

downloader = YoutubeCommentDownloader()

In [ ]:
video_urls = [
    "https://youtu.be/ZftI2fEz0Fw?si=04TADyL6spnef3ot",
    "https://youtu.be/8iAQoRc32rM?si=KUU23O92LjmTCrVa",
    "https://youtu.be/KalbMqA9ARE?si=VMGGDFiCGZHjlYJl",
    "https://youtu.be/IaSv64dC44g?si=DUUp8K6wuKrJgZpX",
    "https://youtu.be/-hKTYvTpYLM?si=fERV2U4FKm1ScTAo",
    "https://youtu.be/SAL-mNE10TA?si=Ea4rjWmdI23KfGT9",
    "https://youtu.be/bnDecJ1X6Wg?si=NQK2X6cg4x369vtT",
    "https://youtu.be/wM-BP6lZyLg?si=dwnz2dKVq7X2OCOY",
    "https://youtu.be/3-jvjdIEy0U?si=cbnMVYR5ZphDSFAV",
    "https://youtu.be/rKNN9t_vTxM?si=mJ17Tqky6sah_JfU",
    "https://youtu.be/MjnFMbhn7f0?si=DOJOXCHf1ejdI44T",
    "https://youtu.be/Cwch6W495GE?si=rdyGim4WA1850QjO",
]

all_comments = []

for url in video_urls:
    count = 0
    for comment in downloader.get_comments_from_url(url, sort_by=SORT_BY_POPULAR):
        all_comments.append({'comment': comment['text'], 'source_url': url})
        count += 1
        if count >= 2000:  # 2000 per video
            break
    print(f"Done: {url} → {count} comments")

df = pd.DataFrame(all_comments)
df.to_csv('raw_comments_multi.csv', index=False)
print(f"Total: {df.shape}")

In [ ]:

df = pd.read_csv('raw_comments_multi.csv')
print("Before cleaning:", df.shape)
print(df.head())

## 2. Data Preprocessing

### 2.1 Initial Cleaning

Raw comments were cleaned in stages:

| Step | Action | Rationale |
|------|--------|-----------|
| Deduplication | Remove exact duplicates | Prevents data leakage |
| Null removal | Drop empty comments | Ensures valid training samples |
| Lowercasing | Convert to lowercase | Reduces vocabulary size |
| URL removal | Strip `http` links | URLs carry no sentiment |
| Mention/hashtag removal | Remove `@user`, `#tag` | Social noise, not sentiment |

### 2.2 Emoji Handling

Emojis were converted to textual descriptions using the `emoji` library (e.g., `😊` → `smiling face`). This preserves sentiment information that would otherwise be lost.

### 2.3 Text Cleaning Strategy

Two approaches were considered:

- **Aggressive cleaning:** Remove all punctuation, numbers, special characters
- **Mild cleaning:** Retain `!`, `?`, `.`, `,`, and numbers

**Choice:** Mild cleaning was selected. Punctuation carries sentiment intensity (`amazing!!!` &gt; `amazing`). Numbers appear in ratings (`0/10`). Aggressive cleaning would strip these cues and reduce accuracy.

### 2.4 Readability Filter

A custom `is_readable` filter removes comments containing only non-ASCII characters (Chinese, Arabic symbols). The threshold requires at least 3 ASCII characters to retain Hinglish romanization.

**Result:** ~13,180 valid comments after all cleaning stages.

In [ ]:
df.info()
# Remove empty comments
df.dropna(subset=['comment'], inplace=True)

# Remove duplicate comments
df.drop_duplicates(subset=['comment'], inplace=True)

print("After removing duplicates & nulls:", df.shape)

##Emoji Handler

In [ ]:
!pip install emoji langdetect

In [ ]:
import emoji

def convert_emojis(text):
    return emoji.demojize(text, delimiters=(" ", " "))

df['comment'] = df['comment'].apply(convert_emojis)
print("Emojis converted to text ")

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)      # remove URLs only
    text = re.sub(r'@\w+', '', text)                # remove mentions
    text = re.sub(r'#\w+', '', text)                # remove hashtags

    # Keep ! ? . , numbers, repeated letters for sentiment
    # Remove only: special symbols, non-ASCII spam
    text = re.sub(r'[^\w\s!?.,0-9]', ' ', text)

    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Re-apply cleaning to original df (before is_readable filter)
df['cleaned_comment'] = df['comment'].apply(clean_text)

# Remove empty rows after cleaning
df = df[df['cleaned_comment'].str.strip() != '']
df = df[df['cleaned_comment'].str.len() > 3]

print("Text cleaned (mild version)")
print(df[['comment', 'cleaned_comment']].head(10))

# Re-apply readable filter
df = df[df['cleaned_comment'].apply(is_readable)]
print("Remaining comments:", df.shape)

In [ ]:
def is_readable(text):
    # Remove comments that are ONLY symbols/chinese/arabic etc
    readable = re.sub(r'[^\x00-\x7F]+', '', text).strip()
    return len(readable) > 3  # must have some english/hinglish characters

df = df[df['cleaned_comment'].apply(is_readable)]
print("Unreadable comments removed ")
print("Remaining comments:", df.shape)
df.to_csv('cleaned_comment.csv', index=False)

In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline
import pandas as pd

# This model is trained specifically on social media/reviews
labeler = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

print("RoBERTa loaded ")

## 3. Sentiment Labeling

### 3.1 Labeling Approach

Manual labeling of 13,000 comments was infeasible. Two automated options were evaluated:

| Approach | Pros | Cons |
|----------|------|------|
| VADER (lexicon-based) | Fast, no GPU needed | Poor on Hinglish, code-mixed text |
| RoBERTa (model-based) | Trained on social media | English-only, may miss Hinglish nuance |

**Choice:** `cardiffnlp/twitter-roberta-base-sentiment-latest` was selected. It is specifically fine-tuned on social media text and provides three labels (positive, negative, neutral) required for this study.

### 3.2 Batch Processing

Comments were processed in batches of 32 to manage GPU memory. Each batch was truncated to 512 tokens (RoBERTa's maximum). Error handling marked failed batches as neutral to prevent pipeline crashes.

**Label distribution (before balancing):**
- Neutral: ~6,800
- Positive: ~3,200
- Negative: ~3,200

In [ ]:

from transformers import pipeline

# Check if we need to re-label
print(f"Current df rows: {len(df)}")
print(f"Sentiment column exists: {'sentiment' in df.columns}")

if 'sentiment' not in df.columns:
    print("Need to label from scratch")
    df['sentiment'] = None
    need_label = True
else:
    print(f"Existing sentiment counts:\n{df['sentiment'].value_counts()}")
    need_label = input("Re-label? (y/n): ").lower() == 'y'

if need_label:
    labeler = pipeline(
        "text-classification",
        model="cardiffnlp/twitter-roberta-base-sentiment-latest",
        tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest"
    )

    labels = []
    batch_size = 32

    for i in range(0, len(df), batch_size):
        batch = df['cleaned_comment'].iloc[i:i+batch_size].tolist()
        batch = [str(t)[:512] for t in batch]
        try:
            results = labeler(batch)
            for r in results:
                label = r['label'].lower()
                if 'pos' in label:
                    labels.append('positive')
                elif 'neg' in label:
                    labels.append('negative')
                else:
                    labels.append('neutral')
        except:
            labels.extend(['neutral'] * len(batch))

        if len(labels) % 500 == 0:
            print(f"Labeled: {len(labels)}/{len(df)}")

    df['sentiment'] = labels
    print("\nLabeling complete!")
    print(df['sentiment'].value_counts())
else:
    print("Using existing labels")

## 4. Dataset Balancing

### 4.1 Imbalance Problem

The raw dataset showed severe class imbalance (neutral &gt; 2× positive/negative). Training on imbalanced data biases models toward the majority class.

### 4.2 Balancing Strategy

Two approaches were considered:

| Method | Description | Risk |
|--------|-------------|------|
| Oversampling (SMOTE) | Duplicate minority samples | Overfitting on synthetic data |
| Undersampling | Reduce majority to minority size | Information loss |

**Choice:** Random undersampling to the smallest class size (~3,200 per class). This was preferred because:
- Social media sentiment is inherently noisy; removing excess neutral samples reduces redundancy
- 9,600 total samples remain sufficient for deep learning
- Prevents synthetic artifacts from SMOTE

**Final dataset:** 9,552 comments (3,184 per class), shuffled with `random_state=42`.

In [ ]:
from sklearn.utils import resample

positive = df[df['sentiment'] == 'positive']
negative = df[df['sentiment'] == 'negative']
neutral  = df[df['sentiment'] == 'neutral']

print("Before balancing:")
print(f"Positive: {len(positive)}")
print(f"Negative: {len(negative)}")
print(f"Neutral:  {len(neutral)}")

min_size = min(len(positive), len(negative), len(neutral))
print(f"\nBalancing to {min_size} each...")

pos_bal = resample(positive, n_samples=min_size, replace=False, random_state=42)
neg_bal = resample(negative, n_samples=min_size, replace=False, random_state=42)
neu_bal = resample(neutral,  n_samples=min_size, replace=False, random_state=42)

df_balanced = pd.concat([pos_bal, neg_bal, neu_bal])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nAfter balancing:")
print(df_balanced['sentiment'].value_counts())
print(f"\nTotal: {len(df_balanced)}")

In [ ]:
df_balanced.to_csv('youtube_hinglish_sentiment_dataset.csv', index=False)
print("Saved as youtube_hinglish_sentiment_dataset.csv ")
df_balanced.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Professional Color Palette

colors = ['#1B2A4A', '#2E5090', '#4A90D9']
bg_color = '#F8F9FA'
text_color = '#1B2A4A'
grid_color = '#DEE2E6'

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.facecolor'] = bg_color
plt.rcParams['figure.facecolor'] = bg_color
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = grid_color
plt.rcParams['grid.alpha'] = 0.7

#Plot 1 — Bar Charts
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor(bg_color)

# Before Balancing
# Use ACTUAL counts from unbalanced df, not hardcoded numbers
before_counts = df['sentiment'].value_counts().to_dict()

bars1 = axes[0].bar(before_counts.keys(), before_counts.values(),
                    color=colors, edgecolor='white',
                    width=0.5, linewidth=1.5)
axes[0].set_title('Before Balancing', fontsize=15,
                   fontweight='bold', color=text_color, pad=15)
axes[0].set_xlabel('Sentiment Class', fontsize=12, color=text_color)
axes[0].set_ylabel('Number of Comments', fontsize=12, color=text_color)
axes[0].tick_params(colors=text_color)
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Value labels on bars
for bar in bars1:
    height = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width()/2., height + 80,
                f'{int(height):,}', ha='center', va='bottom',
                fontweight='bold', fontsize=11, color=text_color)

# After Balancing
after_counts = df_balanced['sentiment'].value_counts()
after_dict = {'Neutral': after_counts['neutral'],
              'Positive': after_counts['positive'],
              'Negative': after_counts['negative']}

bars2 = axes[1].bar(after_dict.keys(), after_dict.values(),
                    color=colors, edgecolor='white',
                    width=0.5, linewidth=1.5)
axes[1].set_title('After Balancing', fontsize=15,
                   fontweight='bold', color=text_color, pad=15)
axes[1].set_xlabel('Sentiment Class', fontsize=12, color=text_color)
axes[1].set_ylabel('Number of Comments', fontsize=12, color=text_color)
axes[1].tick_params(colors=text_color)
axes[1].spines['top'].set_visible(False)
axes[1].spines['right'].set_visible(False)

# Value labels on bars
for bar in bars2:
    height = bar.get_height()
    axes[1].text(bar.get_x() + bar.get_width()/2., height + 30,
                f'{int(height):,}', ha='center', va='bottom',
                fontweight='bold', fontsize=11, color=text_color)

plt.suptitle('YouTube Hinglish Sentiment Dataset Distribution',
             fontsize=17, fontweight='bold',
             color=text_color, y=1.02)
plt.tight_layout()
plt.savefig('dataset_distribution.png', dpi=300, bbox_inches='tight')
plt.show()




In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor(bg_color)

pie_colors = ['#1B2A4A', '#2E5090', '#4A90D9']
explode = (0.05, 0.05, 0.05)

before_counts = df['sentiment'].value_counts().to_dict()

# Before pie
wedges1, texts1, autotexts1 = axes[0].pie(
    before_counts.values(),           # ← use the variable, not assignment
    labels=before_counts.keys(),
    autopct='%1.1f%%',
    colors=pie_colors,
    explode=explode,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for text in texts1:
    text.set_color(text_color)
    text.set_fontsize(12)
for autotext in autotexts1:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(11)
axes[0].set_title('Before Balancing', fontsize=15,
                   fontweight='bold', color=text_color, pad=15)

# After pie
wedges2, texts2, autotexts2 = axes[1].pie(
    after_dict.values(),
    labels=after_dict.keys(),
    autopct='%1.1f%%',
    colors=pie_colors,
    explode=explode,
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for text in texts2:
    text.set_color(text_color)
    text.set_fontsize(12)
for autotext in autotexts2:
    autotext.set_color('white')
    autotext.set_fontweight('bold')
    autotext.set_fontsize(11)
axes[1].set_title('After Balancing', fontsize=15,
                   fontweight='bold', color=text_color, pad=15)

plt.suptitle('Sentiment Distribution — Before vs After',
             fontsize=17, fontweight='bold',
             color=text_color)
plt.tight_layout()
plt.savefig('sentiment_pie_charts.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
df_balanced= pd.read_csv('youtube_hinglish_sentiment_dataset.csv')
print(df_balanced.shape)
print(df_balanced['sentiment'].value_counts())

## 5. Train-Test Split and Encoding

### 5.1 Split Strategy

- **Split ratio:** 80% train, 20% test
- **Stratification:** `stratify=y` ensures identical class distribution in both sets
- **Random state:** 42 for reproducibility

**Result:** 7,641 train samples, 1,911 test samples.

### 5.2 Tokenization

The Keras `Tokenizer` was configured with:
- `num_words=20,000` (increased from initial 10,000 to capture more Hinglish vocabulary)
- `oov_token="&lt;OOV&gt;"` for out-of-vocabulary handling

**Why not pretrained embeddings?** fastText and Word2Vec lack comprehensive Hinglish coverage. Training embeddings from scratch allows the model to learn domain-specific representations (YouTube slang, romanized Hindi).

### 5.3 Sequence Length

Sequence length was set to the 95th percentile of training comment lengths (~55 tokens) plus a 5-token buffer. This covers most comments without excessive padding.

**Padding:** Post-padding with post-truncation. Early tokens (sentence beginnings) are preserved, which carry more sentiment weight in English/Hinglish syntax.

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

le = LabelEncoder()
df_balanced['label'] = le.fit_transform(df_balanced['sentiment'])

print(df_balanced[['sentiment', 'label']].drop_duplicates().sort_values('label'))

X = df_balanced['cleaned_comment']
y = df_balanced['label']

print(f"\nTotal samples: {len(X)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain size: {len(X_train)}")
print(f"Test size:  {len(X_test)}")
print(f"\nTrain distribution:\n{y_train.value_counts()}")
print(f"\nTest distribution:\n{y_test.value_counts()}")

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

vocab_size = 20000  # INCREASED from 10000
oov_token = "<OOV>"

tokenizer = Tokenizer(num_words=vocab_size, oov_token=oov_token)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

print(f"Total unique words: {len(tokenizer.word_index)}")
print(f"Using top {vocab_size} words")
print(f"\nSample: {X_train.iloc[0]}")
print(f"Tokens: {X_train_seq[0][:10]}...")

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Check new 95th percentile
lengths = X_train.apply(lambda x: len(str(x).split()))
print(lengths.describe())

max_len = int(lengths.quantile(0.95)) + 5
print(f"\n95th percentile: {lengths.quantile(0.95):.0f}")
print(f"Using max_len: {max_len}")

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post', truncating='post')

print(f"\nX_train shape: {X_train_pad.shape}")
print(f"X_test shape:  {X_test_pad.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

In [ ]:
with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("Tokenizer saved successfully.")

In [ ]:
import pickle

# Save balanced dataframe
df_balanced.to_csv('/content/drive/My Drive/df_balanced_v2.csv', index=False)

# Save sequences
with open('/content/drive/My Drive/X_train_pad_v2.pkl', 'wb') as f:
    pickle.dump(X_train_pad, f)
with open('/content/drive/My Drive/X_test_pad_v2.pkl', 'wb') as f:
    pickle.dump(X_test_pad, f)

# Save labels
with open('/content/drive/My Drive/y_train_v2.pkl', 'wb') as f:
    pickle.dump(y_train, f)
with open('/content/drive/My Drive/y_test_v2.pkl', 'wb') as f:
    pickle.dump(y_test, f)

# Save tokenizer
with open('/content/drive/My Drive/tokenizer_v2.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("✅ All saved to Google Drive with _v2 suffix:")
print("  - df_balanced_v2.csv")
print("  - X_train_pad_v2.pkl")
print("  - X_test_pad_v2.pkl")
print("  - y_train_v2.pkl")
print("  - y_test_v2.pkl")
print("  - tokenizer_v2.pkl")

## 6. Model Architecture and Training

### 6.1 Common Configuration

All four models share:
- **Embedding layer:** 128-dimensional, vocabulary size 20,000
- **Hidden units:** 64 (balanced capacity vs. overfitting)
- **Output layer:** Dense(3) with softmax for 3-class classification
- **Loss:** `sparse_categorical_crossentropy` (integer labels)
- **Optimizer:** Adam (adaptive learning rate, default parameters)
- **Early stopping:** `patience=3` on validation loss, restore best weights

**Why early stopping on loss, not accuracy?** Validation loss is smoother and less prone to noise-induced fluctuations. Accuracy can plateau while loss continues improving.

### 6.2 Model 1: Simple RNN

**Architecture:** Embedding → SimpleRNN(64) → Dense(3)

**Rationale for inclusion:** RNN serves as the baseline. It is the simplest recurrent architecture and establishes minimum performance expectations.

**Known limitation:** Vanishing gradients in long sequences. Expected to show highest overfitting due to limited memory capacity.

**Training:** 5 epochs before early stopping (validation loss plateau).

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.callbacks import EarlyStopping
import time

model_rnn = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_len),
    SimpleRNN(64),
    Dense(3, activation='softmax')
])

model_rnn.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print("RNN Architecture:")
model_rnn.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("\nTraining RNN...")
start = time.time()

history_rnn = model_rnn.fit(
    X_train_pad, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

rnn_time = time.time() - start

print(f"\n RNN done: {rnn_time:.2f}s, best val: {max(history_rnn.history['val_accuracy']):.4f}")

### 6.3 Model 2: LSTM

**Architecture:** Embedding → LSTM(64) → Dense(3)

**Why LSTM over RNN?** LSTM introduces memory gates (input, forget, output) that mitigate vanishing gradients. This allows learning of long-range dependencies critical for sentiment (e.g., "not bad" requires remembering "not" when evaluating "bad").

**Trade-off:** More parameters → slower training (~17s vs. 9s for RNN), but better generalization expected.

**Training:** 13 epochs before early stopping.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping
import time

model_lstm = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_len),
    LSTM(64),
    Dense(3, activation='softmax')
])

model_lstm.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print("LSTM Architecture:")
model_lstm.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("\nTraining LSTM...")
start = time.time()

history_lstm = model_lstm.fit(
    X_train_pad, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

lstm_time = time.time() - start

print(f"\n LSTM done: {lstm_time:.2f}s, best val: {max(history_lstm.history['val_accuracy']):.4f}")

### 6.4 Model 3: GRU

**Architecture:** Embedding → GRU(64) → Dense(3)

**Why GRU over LSTM?** GRU merges forget and input gates into a single "update" gate, reducing parameters by ~25%. This offers:
- Faster training (fewer matrix operations)
- Comparable memory capacity to LSTM
- Less overfitting risk due to fewer parameters

**When to prefer GRU:** Smaller datasets or when training speed is critical. GRU is often competitive with LSTM on short-to-medium sequences.

**Training:** 9 epochs before early stopping.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense
from tensorflow.keras.callbacks import EarlyStopping
import time

model_gru = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_len),
    GRU(64),
    Dense(3, activation='softmax')
])

model_gru.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print("GRU Architecture:")
model_gru.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("\nTraining GRU...")
start = time.time()

history_gru = model_gru.fit(
    X_train_pad, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

gru_time = time.time() - start

print(f"\n GRU done: {gru_time:.2f}s, best val: {max(history_gru.history['val_accuracy']):.4f}")

### 6.5 Model 4: BiLSTM

**Architecture:** Embedding → Bidirectional(LSTM(64)) → Dense(3)

**Why bidirectional?** Standard LSTM processes left-to-right only. BiLSTM adds a reverse-direction LSTM, capturing context from both sides of each word. This is critical for sentiment where later words modify earlier ones ("was not good" — "not" modifies "good" but appears before it).

**Cost:** 2× parameters (forward + backward LSTM), but only ~1.5× training time due to parallelization.

**Expected advantage:** Best accuracy among RNN variants, especially on ambiguous neutral-class samples requiring full-sentence context.

**Training:** 5 epochs (fastest convergence among all models).

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dense
from tensorflow.keras.callbacks import EarlyStopping
import time

model_bilstm = Sequential([
    Embedding(input_dim=vocab_size, output_dim=128, input_length=max_len),
    Bidirectional(LSTM(64)),
    Dense(3, activation='softmax')
])

model_bilstm.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

print("BiLSTM Architecture:")
model_bilstm.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("\nTraining BiLSTM...")
start = time.time()

history_bilstm = model_bilstm.fit(
    X_train_pad, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

bilstm_time = time.time() - start

print(f"\n BiLSTM done: {bilstm_time:.2f}s, best val: {max(history_bilstm.history['val_accuracy']):.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#F8F9FA')

models_data = [
    ('RNN', history_rnn, '#1B2A4A'),
    ('LSTM', history_lstm, '#2E5090'),
    ('GRU', history_gru, '#4A90D9'),
    ('BiLSTM', history_bilstm, '#6BA3E0')
]

for idx, (name, hist, color) in enumerate(models_data):
    ax = axes.flat[idx]
    ax.set_facecolor('#F8F9FA')
    epochs = range(1, len(hist.history['accuracy']) + 1)
    ax.plot(epochs, hist.history['accuracy'], color=color, linewidth=2.5, marker='o', markersize=5, label='Train Accuracy')
    ax.plot(epochs, hist.history['val_accuracy'], color=color, linewidth=2.5, linestyle='--', marker='s', markersize=5, alpha=0.7, label='Validation Accuracy')
    ax.set_title(f'{name} Training Progress', fontsize=13, fontweight='bold', color='#1B2A4A', pad=10)
    ax.set_xlabel('Epoch', fontsize=11, color='#1B2A4A')
    ax.set_ylabel('Accuracy', fontsize=11, color='#1B2A4A')
    ax.tick_params(colors='#1B2A4A')
    ax.legend(loc='lower right', frameon=True, facecolor='white', edgecolor='#DEE2E6', fontsize=9)
    ax.grid(True, alpha=0.3, color='#DEE2E6')
    ax.set_ylim(0, 1.05)
    for spine in ax.spines.values():
        spine.set_color('#DEE2E6')

plt.suptitle('Model Training Curves: Accuracy vs Epochs', fontsize=16, fontweight='bold', color='#1B2A4A', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/My Drive/chart_training_curves.png', dpi=300, bbox_inches='tight', facecolor='#F8F9FA')
plt.show()

In [ ]:
!pip install transformers torch -q

import torch
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import time

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

# Load tokenizer and model
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=3)
model = model.to(device)

print("DistilBERT loaded!")

In [ ]:
# Tokenize (max 128 for speed)
max_len = 128

def encode(texts):
    return tokenizer(
        texts.tolist(),
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors='pt'
    )

print("Encoding train...")
train_enc = encode(X_train)
train_labels = torch.tensor(y_train.values)

print("Encoding test...")
test_enc = encode(X_test)
test_labels = torch.tensor(y_test.values)

print(f"Train input IDs shape: {train_enc['input_ids'].shape}")

## 7. Transformer Baseline

### 7.1 Why Add DistilBERT?

The abstract specifies four RNN models as the contribution. DistilBERT is included as a **state-of-the-art baseline** to contextualize results, not as a primary contribution.

**Why DistilBERT over BERT?**
- 40% smaller, 60% faster
- Retains 97% of BERT's accuracy
- Fits Colab GPU memory constraints

### 7.2 Architecture

- Pretrained `distilbert-base-uncased` weights
- Fine-tuned with 3-class classification head
- Learning rate: 2e-5 (standard for transformer fine-tuning)
- Batch size: 16 (GPU memory limit)
- Epochs: 3 (transformers converge quickly)

### 7.3 Expected Outcome

DistilBERT should outperform all RNN variants due to self-attention mechanisms that capture global context. The gap quantifies the cost of using lighter RNN architectures.

In [ ]:
from torch.optim import AdamW

# DataLoader
train_dataset = TensorDataset(train_enc['input_ids'], train_enc['attention_mask'], train_labels)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

# Optimizer
optimizer = AdamW(model.parameters(), lr=2e-5)

# Training
model.train()
epochs = 3  # DistilBERT converges fast

print("Training DistilBERT...")
start = time.time()

for epoch in range(epochs):
    total_loss = 0
    for batch in train_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

bert_time = time.time() - start
print(f"\n DistilBERT trained in {bert_time:.2f}s")

In [ ]:
model.eval()

with torch.no_grad():
    test_enc_device = {k: v.to(device) for k, v in test_enc.items()}
    outputs = model(**test_enc_device)
    predictions = torch.argmax(outputs.logits, dim=-1).cpu().numpy()

acc = (predictions == y_test.values).mean()
print(f"DistilBERT Test Accuracy: {acc:.4f}")

In [ ]:
import pickle

# Save model
model.save_pretrained('/content/drive/My Drive/distilbert_sentiment')
tokenizer.save_pretrained('/content/drive/My Drive/distilbert_sentiment')

# Save predictions for report
with open('/content/drive/My Drive/distilbert_predictions.pkl', 'wb') as f:
    pickle.dump(predictions, f)

print("DistilBERT saved to Google Drive!")

## 8. Evaluation Methodology

### 8.1 Metrics

| Metric | Purpose |
|--------|---------|
| Accuracy | Overall correctness |
| Precision | Reliability of positive predictions |
| Recall | Coverage of actual positives |
| F1-Score | Harmonic mean of precision and recall |
| Training time | Efficiency for deployment |
| Memory usage | Hardware requirements |

**Why weighted averages?** Class imbalance (though balanced in training) can reappear in test splits. Weighted metrics account for per-class sample sizes.

### 8.2 Test Set Evaluation

All models were evaluated on the held-out test set (never seen during training). Predictions were generated in a single forward pass to ensure fair time comparison.

### 8.3 Memory Measurement

Peak memory was measured using `psutil` during test prediction. This reflects inference cost, which is critical for deployment on resource-constrained devices.

## 9. Results and Analysis

### 9.1 Training Behavior

RNN exhibited severe overfitting: training accuracy reached 90% while validation stalled at 65%. This confirms SimpleRNN's inability to generalize from limited context windows.

LSTM and GRU showed healthier convergence with smaller train-validation gaps (~10-15%). GRU trained faster than LSTM despite comparable accuracy, validating its parameter efficiency.

BiLSTM achieved the best validation accuracy (74%) with the fastest convergence (5 epochs), demonstrating the value of bidirectional context.

### 9.2 Test Set Performance

**Table 1:** Complete model comparison (accuracy, precision, recall, F1, time, memory)

*(Insert `table_complete_comparison.png` here)*

**Key findings:**
- BiLSTM: Best RNN variant (74% accuracy, 9.9s training)
- DistilBERT: Highest overall (85% accuracy, 250s training)
- GRU: Most efficient (72.5% accuracy, 13.4s training)
- RNN: Baseline only (65% accuracy, severely overfitted)

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import psutil
import os

# Function to get memory usage in MB
def get_memory_mb():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

# Your 4 RNN models
models_rnn = {
    'RNN': model_rnn,
    'LSTM': model_lstm,
    'GRU': model_gru,
    'BiLSTM': model_bilstm
}

histories = {
    'RNN': history_rnn,
    'LSTM': history_lstm,
    'GRU': history_gru,
    'BiLSTM': history_bilstm
}

times = {
    'RNN': rnn_time,
    'LSTM': lstm_time,
    'GRU': gru_time,
    'BiLSTM': bilstm_time
}

results = []

print("="*80)
print("FINAL EVALUATION: ALL MODELS ON TEST SET")
print("="*80)

# Evaluate 4 RNN models
for name, model in models_rnn.items():
    print(f"\n{'-'*60}")
    print(f"Model: {name}")
    print(f"{'-'*60}")

    # Memory before prediction
    mem_before = get_memory_mb()

    y_pred = model.predict(X_test_pad, verbose=0).argmax(axis=1)

    # Memory after prediction
    mem_after = get_memory_mb()
    mem_used = mem_after - mem_before

    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True, target_names=['negative', 'neutral', 'positive'])

    cm = confusion_matrix(y_test, y_pred)
    print(f"Confusion Matrix:\n{cm}")
    print(f"Memory used: {mem_used:.2f} MB")

    results.append({
        'Model': name,
        'Test Accuracy': round(acc * 100, 2),
        'Precision': round(report['weighted avg']['precision'] * 100, 2),
        'Recall': round(report['weighted avg']['recall'] * 100, 2),
        'F1-Score': round(report['weighted avg']['f1-score'] * 100, 2),
        'Train Time (s)': round(times[name], 2),
        'Memory (MB)': round(mem_used, 2),
        'Epochs': len(histories[name].history['loss']),
        'Best Val Acc (%)': round(max(histories[name].history['val_accuracy']) * 100, 2),
        'Architecture Type': 'RNN Variant'
    })

    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['negative', 'neutral', 'positive']))

# Evaluate DistilBERT
print(f"\n{'-'*60}")
print(f"Model: DistilBERT")
print(f"{'-'*60}")

mem_before = get_memory_mb()

# DistilBERT prediction (already have predictions from earlier)
bert_acc = accuracy_score(y_test, predictions)
bert_report = classification_report(y_test, predictions, output_dict=True, target_names=['negative', 'neutral', 'positive'])

mem_after = get_memory_mb()
mem_used = mem_after - mem_before

results.append({
    'Model': 'DistilBERT',
    'Test Accuracy': round(bert_acc * 100, 2),
    'Precision': round(bert_report['weighted avg']['precision'] * 100, 2),
    'Recall': round(bert_report['weighted avg']['recall'] * 100, 2),
    'F1-Score': round(bert_report['weighted avg']['f1-score'] * 100, 2),
    'Train Time (s)': round(bert_time, 2),
    'Memory (MB)': round(mem_used, 2),
    'Epochs': 3,
    'Best Val Acc (%)': '-',
    'Architecture Type': 'Transformer'
})

print(f"Memory used: {mem_used:.2f} MB")
print(f"\nClassification Report:")
print(classification_report(y_test, predictions, target_names=['negative', 'neutral', 'positive']))

# Create DataFrame
results_df = pd.DataFrame(results)


fig, ax = plt.subplots(figsize=(16, 6))
fig.patch.set_facecolor('#F8F9FA')

ax.axis('tight')
ax.axis('off')

table_data = results_df[['Model', 'Test Accuracy', 'Precision', 'Recall', 'F1-Score', 'Train Time (s)', 'Memory (MB)', 'Epochs', 'Architecture Type']].values
col_labels = ['Model', 'Test Acc (%)', 'Precision (%)', 'Recall (%)', 'F1 (%)', 'Time (s)', 'Memory (MB)', 'Epochs', 'Type']

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    cellLoc='center',
    loc='center',
    colColours=['#1B2A4A'] * len(col_labels),
    colWidths=[0.10, 0.10, 0.10, 0.10, 0.08, 0.08, 0.10, 0.08, 0.10]
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

for i in range(len(col_labels)):
    table[(0, i)].set_text_props(color='white', fontweight='bold')
    table[(0, i)].set_fontsize(11)

for i in range(1, len(table_data) + 1):
    for j in range(len(col_labels)):
        table[(i, j)].set_facecolor('#F8F9FA')
        table[(i, j)].set_edgecolor('#DEE2E6')
        if j == 1:  # Test Accuracy column
            table[(i, j)].set_text_props(fontweight='bold')

best_idx = results_df['Test Accuracy'].idxmax() + 1
for j in range(len(col_labels)):
    table[(best_idx, j)].set_facecolor('#D4EDDA')

plt.title('Complete Model Comparison: Accuracy, Time & Memory',
          fontsize=16, fontweight='bold', color='#1B2A4A', pad=20)
plt.savefig('/content/drive/My Drive/table_complete_comparison.png', dpi=300, bbox_inches='tight', facecolor='#F8F9FA')
plt.show()



fig, ax = plt.subplots(figsize=(12, 5))
fig.patch.set_facecolor('#F8F9FA')

ax.axis('tight')
ax.axis('off')

eff_data = results_df[['Model', 'Train Time (s)', 'Memory (MB)', 'Test Accuracy']].values
eff_cols = ['Model', 'Train Time (s)', 'Memory (MB)', 'Test Acc (%)']

table2 = ax.table(
    cellText=eff_data,
    colLabels=eff_cols,
    cellLoc='center',
    loc='center',
    colColours=['#1B2A4A'] * len(eff_cols),
    colWidths=[0.25, 0.25, 0.25, 0.25]
)

table2.auto_set_font_size(False)
table2.set_fontsize(12)
table2.scale(1, 2.5)

for i in range(len(eff_cols)):
    table2[(0, i)].set_text_props(color='white', fontweight='bold')

for i in range(1, len(eff_data) + 1):
    for j in range(len(eff_cols)):
        table2[(i, j)].set_facecolor('#F8F9FA')
        table2[(i, j)].set_edgecolor('#DEE2E6')

plt.title('Efficiency Comparison: Time & Memory Usage',
          fontsize=16, fontweight='bold', color='#1B2A4A', pad=20)
plt.savefig('/content/drive/My Drive/table_efficiency.png', dpi=300, bbox_inches='tight', facecolor='#F8F9FA')
plt.show()

# Save CSV
results_df.to_csv('/content/drive/My Drive/final_comparison_all_models.csv', index=False)
print("\n All tables saved to Google Drive!")
print("Files saved:")
print("  - table_complete_comparison.png")
print("  - table_efficiency.png")
print("  - final_comparison_all_models.csv")

### 9.3 Efficiency Trade-offs

**Figure 1:** Results dashboard combining accuracy bars, time scatter, confusion matrices, and key metrics

*(Insert `chart_dashboard.png` here)*

The scatter plot reveals three clusters:
- **Fast, moderate accuracy:** RNN, GRU, BiLSTM (<15s, 65-74%)
- **Slow, high accuracy:** DistilBERT (250s, 85%)

For real-time applications (live comment moderation), BiLSTM offers the best accuracy-time ratio. For offline batch processing, DistilBERT justifies its cost.

### 9.4 Per-Class Analysis

Confusion matrices reveal class-specific patterns:
- **Negative/Positive:** Better separation (clear sentiment words)
- **Neutral:** Most misclassified (ambiguous intensity, mixed language)

BiLSTM and DistilBERT both struggle with neutral class, suggesting this is a dataset limitation rather than architectural weakness.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

# Generate predictions if not available
if 'preds' not in globals():
    preds = {
        'rnn': model_rnn.predict(X_test_pad, verbose=0).argmax(axis=1),
        'lstm': model_lstm.predict(X_test_pad, verbose=0).argmax(axis=1),
        'gru': model_gru.predict(X_test_pad, verbose=0).argmax(axis=1),
        'bilstm': model_bilstm.predict(X_test_pad, verbose=0).argmax(axis=1),
        'distilbert': predictions if 'predictions' in globals() else np.zeros(len(y_test))
    }

fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#F8F9FA')

# Main bar chart (top, wide)
ax1 = plt.subplot2grid((3, 3), (0, 0), colspan=2)
models = results_df['Model'].tolist()
accuracies = results_df['Test Accuracy'].tolist()
colors = ['#1B2A4A', '#2E5090', '#4A90D9', '#6BA3E0', '#2E5090']
bars = ax1.bar(models, accuracies, color=colors, edgecolor='white', linewidth=2, width=0.5)
for bar, acc in zip(bars, accuracies):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height + 1, f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11, color='#1B2A4A')
ax1.set_title('Test Accuracy Comparison', fontsize=14, fontweight='bold', color='#1B2A4A', pad=10)
ax1.set_ylabel('Accuracy (%)', fontsize=11, color='#1B2A4A')
ax1.set_ylim(0, 100)
ax1.tick_params(colors='#1B2A4A')
ax1.grid(True, axis='y', alpha=0.3, color='#DEE2E6')
for spine in ax1.spines.values():
    spine.set_visible(False)
ax1.spines['left'].set_visible(True)
ax1.spines['bottom'].set_visible(True)

# Time scatter (top right)
ax2 = plt.subplot2grid((3, 3), (0, 2))
times_list = results_df['Train Time (s)'].tolist()
ax2.scatter(times_list, accuracies, s=200, c=colors, edgecolors='white', linewidths=2, alpha=0.8)
for i, name in enumerate(models):
    ax2.annotate(name, (times_list[i], accuracies[i]), xytext=(5, 5), textcoords='offset points', fontsize=8, color='#1B2A4A')
ax2.set_title('Time vs Accuracy', fontsize=12, fontweight='bold', color='#1B2A4A', pad=10)
ax2.set_xlabel('Time (s)', fontsize=9, color='#1B2A4A')
ax2.set_ylabel('Acc (%)', fontsize=9, color='#1B2A4A')
ax2.tick_params(colors='#1B2A4A', labelsize=8)
ax2.grid(True, alpha=0.3, color='#DEE2E6')

# BiLSTM confusion (bottom left)
ax3 = plt.subplot2grid((3, 3), (1, 0), rowspan=2)
cm_bilstm = confusion_matrix(y_test, preds['bilstm'])
sns.heatmap(cm_bilstm, annot=True, fmt='d', cmap='Blues', xticklabels=['Neg', 'Neu', 'Pos'], yticklabels=['Neg', 'Neu', 'Pos'], ax=ax3, cbar_kws={'shrink': 0.6}, annot_kws={'size': 11, 'weight': 'bold'})
ax3.set_title('BiLSTM Confusion Matrix', fontsize=12, fontweight='bold', color='#1B2A4A', pad=10)
ax3.set_xlabel('Predicted', fontsize=10, color='#1B2A4A')
ax3.set_ylabel('True', fontsize=10, color='#1B2A4A')

# DistilBERT confusion (bottom middle)
ax4 = plt.subplot2grid((3, 3), (1, 1), rowspan=2)
cm_bert = confusion_matrix(y_test, preds['distilbert'])
sns.heatmap(cm_bert, annot=True, fmt='d', cmap='Blues', xticklabels=['Neg', 'Neu', 'Pos'], yticklabels=['Neg', 'Neu', 'Pos'], ax=ax4, cbar_kws={'shrink': 0.6}, annot_kws={'size': 11, 'weight': 'bold'})
ax4.set_title('DistilBERT Confusion Matrix', fontsize=12, fontweight='bold', color='#1B2A4A', pad=10)
ax4.set_xlabel('Predicted', fontsize=10, color='#1B2A4A')
ax4.set_ylabel('True', fontsize=10, color='#1B2A4A')

# Key metrics table (bottom right, text)
ax5 = plt.subplot2grid((3, 3), (1, 2), rowspan=2)
ax5.axis('off')

table_text = """
KEY FINDINGS

Best RNN Variant: BiLSTM
  Accuracy: 74.0%
  Time: 9.9s
  Epochs: 5

Transformer Baseline: DistilBERT
  Accuracy: 85.0%
  Time: 250s
  Gain: +11%

Efficiency Winner: GRU
  Accuracy: 72.5%
  Time: 13.4s
  Best speed/accuracy ratio
"""

ax5.text(0.1, 0.5, table_text, transform=ax5.transAxes, fontsize=11, verticalalignment='center', color='#1B2A4A', family='monospace', bbox=dict(boxstyle='round', facecolor='white', edgecolor='#DEE2E6', pad=1))

plt.suptitle('Sentiment Analysis Results Dashboard', fontsize=18, fontweight='bold', color='#1B2A4A', y=1.02)
plt.tight_layout()
plt.savefig('/content/drive/My Drive/chart_dashboard.png', dpi=300, bbox_inches='tight', facecolor='#F8F9FA')
plt.show()

In [ ]:
import pickle

# Check what variables exist
print("Checking variables...")

# Save Keras models
model_rnn.save('/content/drive/My Drive/model_rnn.keras')
model_lstm.save('/content/drive/My Drive/model_lstm.keras')
model_gru.save('/content/drive/My Drive/model_gru.keras')
model_bilstm.save('/content/drive/My Drive/model_bilstm.keras')

print("Keras models saved")

# Try to find DistilBERT model
try:
    distilbert_model.save_pretrained('/content/drive/My Drive/distilbert_sentiment')
    distilbert_tokenizer.save_pretrained('/content/drive/My Drive/distilbert_sentiment')
    print("DistilBERT saved")
except NameError:
    print("distilbert_model not found, trying 'model'...")
    try:
        model.save_pretrained('/content/drive/My Drive/distilbert_sentiment')
        tokenizer.save_pretrained('/content/drive/My Drive/distilbert_sentiment')
        print("DistilBERT saved using 'model'")
    except:
        print("Could not save DistilBERT - model variable name issue")

# Save histories
with open('/content/drive/My Drive/histories.pkl', 'wb') as f:
    pickle.dump({
        'rnn': history_rnn.history,
        'lstm': history_lstm.history,
        'gru': history_gru.history,
        'bilstm': history_bilstm.history
    }, f)

# Save variables
with open('/content/drive/My Drive/variables.pkl', 'wb') as f:
    pickle.dump({
        'vocab_size': vocab_size,
        'max_len': max_len,
        'rnn_time': rnn_time,
        'lstm_time': lstm_time,
        'gru_time': gru_time,
        'bilstm_time': bilstm_time,
        'bert_time': bert_time
    }, f)

results_df.to_csv('/content/drive/My Drive/final_results.csv', index=False)

with open('/content/drive/My Drive/predictions.pkl', 'wb') as f:
    pickle.dump({
        'rnn': model_rnn.predict(X_test_pad, verbose=0).argmax(axis=1),
        'lstm': model_lstm.predict(X_test_pad, verbose=0).argmax(axis=1),
        'gru': model_gru.predict(X_test_pad, verbose=0).argmax(axis=1),
        'bilstm': model_bilstm.predict(X_test_pad, verbose=0).argmax(axis=1),
        'distilbert': predictions
    }, f)

print("\nDone saving!")

## 10. Discussion

### 10.1 Why RNN Variants Underperform Transformers

The 11% accuracy gap (BiLSTM 74% vs. DistilBERT 85%) stems from:
1. **Local context only:** RNNs process sequentially; transformers attend globally
2. **Fixed embeddings:** RNN embeddings are static post-training; BERT embeddings are context-dependent
3. **Pretraining:** DistilBERT leverages 40GB of pretraining data; RNNs start from random initialization

### 10.2 Why BiLSTM is Still Valuable

Despite lower accuracy, BiLSTM offers:
- **28× faster training** (9.9s vs. 250s)
- **Lower memory footprint** (no attention mechanism overhead)
- **Deterministic behavior** (no randomness in attention heads)
- **Easier deployment** (no transformer framework dependencies)

### 10.3 Limitations

- **Dataset size:** 9,600 samples is small for deep learning; 50,000+ would improve all models
- **Label noise:** RoBERTa labels on Hinglish are imperfect; manual verification would help
- **No pretrained Hinglish embeddings:** fastText Hindi-English vectors could boost RNN performance
- **Single dataset:** Results may not generalize to other platforms (Twitter, Instagram)

### 10.4 Future Work

- Multilingual BERT (mBERT, XLM-R) for better Hinglish handling
- Data augmentation via back-translation
- Attention visualization to interpret model decisions
- Real-time deployment on edge devices


## 11. Conclusion

This study systematically compared four RNN architectures for Hinglish sentiment analysis. Key conclusions:

1. **BiLSTM is the best RNN variant** (74% accuracy, fastest convergence, smallest overfitting gap)
2. **GRU offers the best efficiency** (72.5% accuracy, 13.4s, best speed-accuracy trade-off)
3. **Simple RNN is insufficient** for this task (65% accuracy, severe overfitting)
4. **Transformers dominate accuracy** but at significant computational cost

For practitioners choosing between architectures:
- **Resource-constrained deployment:** GRU
- **Best accuracy with RNN constraints:** BiLSTM
- **Maximum accuracy acceptable:** DistilBERT

The code, models, and datasets are fully reproducible and available in this notebook.

---

**References**

1. Hochreiter, S., & Schmidhuber, J. (1997). Long short-term memory. Neural Computation.
2. Cho, K., et al. (2014). Learning phrase representations using RNN encoder-decoder. EMNLP.
3. Sanh, V., et al. (2019). DistilBERT, a distilled version of BERT. arXiv.
4. Cardiff NLP. (2021). Twitter-RoBERTa for sentiment analysis. HuggingFace.